# 03 - Embeddings And Similarity

Embeddings are vector representations of text. They let software compare meaning with math and are used in search, recommendation, clustering, duplicate detection, routing, and RAG.


## Why Similarity Needs Vectors

Embeddings turn text into vectors so software can compare meaning mathematically. They are central to semantic search, recommendations, clustering, deduplication, and retrieval-augmented generation. A vector does not contain truth; it contains a position in a representation space created by a model or vectorizer.

This notebook uses a transparent TF-IDF-style vectorizer instead of a provider embedding model. That makes the ranking behavior easier to inspect before introducing stronger black-box embeddings.

**From text to vectors.**

An embedding is a numeric representation that lets software compare pieces of text mathematically. In modern LLM systems, embeddings are often dense vectors produced by neural models. This notebook uses a transparent TF-IDF-style vectorizer instead, because it exposes the mechanics: tokenize documents, identify useful terms, weight rare terms more strongly than common terms, normalize vectors, and compare them with cosine similarity.

The transparent version is less powerful than a neural embedding model, but it teaches the same retrieval workflow. Every semantic search system needs document preparation, vector creation, query vector creation, similarity scoring, ranking, and result inspection.


## What Cosine Similarity Means

Cosine similarity compares vector direction rather than raw length. Two texts receive a high score when their weighted features point in a similar direction. This is useful because a long document should not win only because it has more words. The score is relative to the corpus, preprocessing choices, and vectorizer. A score of 0.35 can be strong in one setup and weak in another, so always inspect examples rather than treating the number as universal truth.


## Objectives

- Build local TF-IDF-style text vectors.
- Compare a query with documents using cosine similarity.
- Inspect why a result ranked first.
- Explain why similarity scores are relative, not universal truth.


```mermaid
flowchart TD
    A[Text documents] --> B[Preprocess]
    B --> C[Vectorize]
    C --> D[Compare with query]
    D --> E[Rank by similarity]
    E --> F[Inspect top results]
```


## From Keywords To Semantic Search

Embeddings are useful because exact keywords are often too brittle. A support user may ask “How can I make this cheaper?” while the relevant lesson says “token-based cost estimation.” A pure keyword system might miss that relationship. A stronger embedding model can often connect the ideas even when the wording differs.

This notebook uses a transparent TF-IDF-style vectorizer, not a neural embedding model. That is intentional: it lets you inspect the mechanics before treating embeddings as a black box.


**Before reading the helper cell:** follow the same steps a library would hide: tokenize, weight terms, normalize vectors, and compare directions.

The helper cell implements the pieces that many libraries hide. `content_tokens` removes simple stopwords so common words do not dominate. `fit_idf` measures how rare each token is across documents. `vectorize` combines term frequency with inverse document frequency and normalizes the result. `cosine_similarity` then compares only shared vector dimensions. Reading this code from top to bottom mirrors the lifecycle of a small search index.


In [ ]:
# Local vector helpers are defined here so practitioners can inspect every step.
# This TF-IDF-style approach is transparent; production embeddings are usually neural models.
import csv
import math
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

TOKEN_PATTERN = re.compile(
    r"""
    [A-Za-z]+(?:'[A-Za-z]+)?  # words and contractions
    |\d+(?:\.\d+)?           # integers and decimals
    |[^\w\s]                 # punctuation and symbols
    """,
    re.VERBOSE,
)


@dataclass(frozen=True)
class TokenizationResult:
    text: str
    tokens: list[str]

    @property
    def count(self) -> int:
        return len(self.tokens)


def simple_tokenize(text: str) -> list[str]:
    """Split text into approximate tokens for transparent local practice."""
    return TOKEN_PATTERN.findall(text)


def inspect_text(text: str) -> TokenizationResult:
    return TokenizationResult(text=text, tokens=simple_tokenize(text))


def count_tokens(text: str) -> int:
    return len(simple_tokenize(text))


def token_budget_remaining(
    context_window: int, prompt_tokens: int, reserved_output_tokens: int
) -> int:
    return context_window - prompt_tokens - reserved_output_tokens

### Text cleanup and feature selection

The next block decides which tokens are informative enough to compare. It removes simple stopwords, combines useful metadata fields into searchable text, and prepares the terms that will later receive TF-IDF weights.


In [ ]:
# Sparse vectors store only terms that appear, which keeps the representation readable.
SparseVector = dict[str, float]

# Remove common glue words so topic words carry more of the score.
STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "as",
    "be",
    "by",
    "can",
    "do",
    "does",
    "for",
    "from",
    "how",
    "i",
    "in",
    "into",
    "is",
    "it",
    "me",
    "my",
    "of",
    "or",
    "should",
    "the",
    "this",
    "to",
    "what",
    "when",
    "whether",
    "why",
    "with",
}

### Vector construction and comparison

Once text has been cleaned, the remaining functions fit inverse document-frequency weights, build normalized sparse vectors, and compare vectors with cosine similarity. This is the core of the local embedding approximation used in the rest of the notebook.


In [ ]:
# Bundle records, learned IDF weights, and vectors so later cells cannot mix mismatched objects.
@dataclass(frozen=True)
class EmbeddedCorpus:
    records: list[dict[str, str]]
    text_field: str
    idf: dict[str, float]
    vectors: list[SparseVector]


def content_tokens(text: str) -> list[str]:
    """Return comparable lowercase tokens after removing low-signal words."""
    tokens = [token.lower() for token in simple_tokenize(text)]
    return [
        token
        for token in tokens
        if token not in STOPWORDS and any(char.isalnum() for char in token)
    ]


def combine_text_fields(record: dict[str, str], fields: list[str]) -> str:
    """Join selected metadata fields into the text that will be vectorized."""
    missing = [field for field in fields if field not in record]
    if missing:
        raise ValueError(f"Missing expected fields: {missing}")
    return " ".join(record.get(field, "") for field in fields)


def fit_idf(texts: list[str]) -> dict[str, float]:
    """Learn how informative each token is across the local catalog."""
    document_count = len(texts)
    document_frequency: Counter[str] = Counter()
    for text in texts:
        document_frequency.update(set(content_tokens(text)))
    return {
        token: math.log((1 + document_count) / (1 + frequency)) + 1
        for token, frequency in document_frequency.items()
    }

### Vector construction and comparison

With document weights available, the next block builds normalized vectors and compares their direction. Keeping this separate from text cleanup makes the retrieval pipeline easier to debug.


In [ ]:
def vectorize(text: str, idf: dict[str, float]) -> SparseVector:
    """Convert text into a normalized TF-IDF vector for cosine comparison."""
    tokens = content_tokens(text)
    if not tokens:
        return {}
    counts = Counter(tokens)
    total = len(tokens)
    vector = {
        token: (count / total) * idf.get(token, 0.0)
        for token, count in counts.items()
        if token in idf
    }
    norm = math.sqrt(sum(value * value for value in vector.values()))
    if norm == 0:
        return vector
    return {token: value / norm for token, value in vector.items()}


def embed_corpus(records: list[dict[str, str]], text_field: str) -> EmbeddedCorpus:
    """Fit vector weights once, then vectorize every record with the same weights."""
    if not records:
        raise ValueError("Cannot embed an empty corpus.")
    texts = [record[text_field] for record in records]
    idf = fit_idf(texts)
    vectors = [vectorize(text, idf) for text in texts]
    return EmbeddedCorpus(
        records=records, text_field=text_field, idf=idf, vectors=vectors
    )


def cosine_similarity(left: SparseVector, right: SparseVector) -> float:
    """Compare vector direction; empty vectors receive no similarity credit."""
    if not left or not right:
        return 0.0
    shared = set(left).intersection(right)
    return sum(left[token] * right[token] for token in shared)


def similarity_scores(corpus: EmbeddedCorpus, query: str) -> list[float]:
    """Score one query against every document vector in the fitted corpus."""
    query_vector = vectorize(query, corpus.idf)
    return [cosine_similarity(query_vector, vector) for vector in corpus.vectors]

## Load The Course Catalog

Each row is a small document. The searchable text combines title, module, and description so retrieval can use both metadata and content.


**Before loading the data:** notice how separate CSV fields become one searchable text field while the original metadata stays available for display.

The catalog CSV has separate fields for lesson ID, title, module, and description. The code combines title, module, and description into one `search_text` field because retrieval needs both metadata and explanatory content. Keeping the original fields allows later cells to display readable titles and modules instead of anonymous vectors.


In [ ]:
# Load local CSV data; keeping data on disk makes the examples reproducible.
catalog_path = Path("data") / "course_catalog.csv"
with catalog_path.open(newline="", encoding="utf-8") as file:
    records = list(csv.DictReader(file))
for record in records:
    record["search_text"] = combine_text_fields(
        record, ["title", "module", "description"]
    )
print(f"Loaded {len(records)} course lessons")
for record in records:
    print(f"- {record['lesson_id']}: {record['title']}")

## Build Local Text Vectors

This vectorizer is deliberately transparent. Production embedding models are stronger, but the workflow is the same: convert text to vectors, compare vectors, inspect the ranked results.


## Embedding System Anatomy

```mermaid
flowchart TD
    A[Raw documents] --> B[Clean and combine fields]
    B --> C[Chunk or keep short records]
    C --> D[Create vectors]
    D --> E[Store vectors with metadata]
    F[User query] --> G[Create query vector]
    G --> H[Compare query to documents]
    E --> H
    H --> I[Return ranked results]
```

Metadata is part of the system, not decoration. Titles, modules, source URLs, timestamps, and document IDs help users and developers interpret why a result appeared.


In [ ]:
# Fit vectors once over the catalog before comparing individual queries.
corpus = embed_corpus(records, text_field="search_text")
print(f"Vocabulary size: {len(corpus.idf)}")
print("Sample terms:", sorted(corpus.idf)[:20])

## Rank Lessons For A Query

Cosine similarity compares vector direction. Higher scores mean the texts point in a more similar direction in this local vector space.


In [ ]:
# Rank every catalog row against one query, then inspect the strongest matches.
query = "How can I estimate the price of an LLM request?"
scores = similarity_scores(corpus, query)
ranked = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
print(f"Query: {query}\n")
for rank, index in enumerate(ranked[:3], start=1):
    row = records[index]
    print(f"{rank}. {row['title']} | score={scores[index]:.3f}")
    print(f"   {row['description'][:150]}...")

## Inspect Matching Terms

Simple vector systems are easier to debug because shared content tokens are visible. Neural embeddings are less transparent, but result inspection is still required.


In [ ]:
# Compare query terms with the top document to debug why it ranked first.
top_index = ranked[0]
query_terms = set(content_tokens(query))
doc_terms = set(content_tokens(records[top_index]["search_text"]))
print("Query terms:", sorted(query_terms))
print("Shared terms with top result:", sorted(query_terms & doc_terms))

## Score Interpretation

A cosine similarity score is meaningful only inside a specific setup: the documents, vectorizer, preprocessing choices, query style, and chunk length. A score of `0.30` may be strong in one corpus and weak in another.

For real projects, inspect ranked examples instead of choosing a threshold blindly. Look at successful queries, ambiguous queries, unrelated queries, and cases where the relevant document appears at rank 2 or rank 3.


## When To Use A Library Instead Of This Local Vectorizer

Use the local vectorizer when the goal is inspection, teaching, or a tiny reproducible example. Move to a library when you need better semantic matching, multilingual support, larger document collections, persistence, metadata filtering, or fast nearest-neighbor search. A practical progression is: first prove the retrieval workflow locally, then replace only the vectorization or storage layer with a production tool while keeping the same evaluation questions.


## Compare Related And Unrelated Queries

Embedding systems always return nearest neighbors unless you add a fallback rule. That is useful for ranking, but dangerous if the top score is weak.


In [ ]:
# Include an unrelated query to show why nearest-neighbor search needs fallback policy.
queries = [
    "How do I count tokens?",
    "How do I estimate LLM cost?",
    "How do I bake bread?",
]
for query in queries:
    scores = similarity_scores(corpus, query)
    best_index = max(range(len(scores)), key=lambda index: scores[index])
    print(f"{query!r} -> {records[best_index]['title']} ({scores[best_index]:.3f})")

## What To Inspect

Do not treat the score as a universal confidence value. Compare top-1, top-3, and failure cases. Ask whether the result actually helps answer the query.


## Practice

Change `your_query` and rerun the cell. Try one attention question, one hallucination question, and one unrelated question.


## Common Embedding Failure Modes

| Failure mode | What it looks like | First fix to try |
| --- | --- | --- |
| Keyword mismatch | Relevant result does not appear because wording differs. | Try a stronger embedding model or add clearer document text. |
| Semantic drift | Result is broadly related but not useful. | Improve chunking and evaluate top-k examples. |
| Chunk boundary issue | Answer is split across chunks. | Add overlap or use section-aware chunking. |
| Weak metadata | Result cannot be traced or cited. | Store source, title, section, and update date. |

The goal is not just to retrieve something. The goal is to retrieve context that helps the next system step make a grounded decision.


In [ ]:
# Editable practice query: rerun this cell to inspect how rankings change.
your_query = "Why do chatbots hallucinate?"
your_scores = similarity_scores(corpus, your_query)
your_ranked = sorted(
    range(len(your_scores)), key=lambda index: your_scores[index], reverse=True
)
for rank, index in enumerate(your_ranked[:3], start=1):
    print(f"{rank}. {records[index]['title']} | score={your_scores[index]:.3f}")

## Practice Reference Answer

A good run should show that course-related queries retrieve relevant modules, while an unrelated query still receives a nearest neighbor. The correct interpretation is that similarity search ranks candidates; it does not decide whether the query belongs in scope. A production workflow therefore needs score inspection, source review, and fallback behavior.

## Checkpoint

Answer the questions, then compare with the reference answers below:

1. What does an embedding represent?
2. Why is cosine similarity useful for text retrieval?
3. Why can an unrelated query still return a result?
4. What would you inspect before trusting an embedding search system?


## Checkpoint Reference Answers

1. An embedding represents text as a numeric vector whose position captures patterns useful for comparison.
2. Cosine similarity is useful because it compares vector direction, which helps rank texts that are semantically close even when their length differs.
3. An unrelated query still returns a result because nearest-neighbor search ranks the closest available document even when every candidate is weak.
4. Inspect the top-k results, shared terms or explanations when available, score distribution, source metadata, failure cases, and whether a fallback threshold is needed.